## 1. Load the data
Read `AB_NYC_2019.csv` into a DataFrame called `df`, then report how many rows and columns it has and what data type pandas inferred for each column (e.g. `int64` for whole numbers, `float64` for decimals, `str`/`object` for text).

In [ ]:
import pandas as pd

df = pd.read_csv("AB_NYC_2019.csv")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
df.dtypes

## 2. Preview the data
Look at the first 10 rows to sanity-check that the file loaded correctly and the columns contain what we expect.

In [ ]:
df.head(10)

## 3. Check for fully duplicate rows
A "fully duplicate" row is one where *every* column matches another row exactly (not just the `id`). `df.duplicated(keep=False)` flags all rows that participate in such a duplicate pair/group (both the original and its copy), so we can see:
- how many duplicate rows exist,
- which row numbers (DataFrame index) they are,
- and, for context, every row sharing that same listing `id` (even if not a full duplicate) so we can compare them side by side.

In [ ]:
# Full duplicate rows: show their row numbers and the matching row(s) sharing the same id
dupe_mask = df.duplicated(keep=False)
dupe_rows = df[dupe_mask]
print(f"Fully duplicate rows: {df.duplicated().sum()}")
print(f"Row numbers involved in duplication: {dupe_rows.index.tolist()}")

if not dupe_rows.empty:
    for dup_id in dupe_rows["id"].unique():
        display(df[df["id"] == dup_id])
else:
    print("No duplicate rows found.")

## 4. Convert `last_review` from text to a real date
Right now `last_review` is stored as plain text (`str`/`object`), so pandas cannot do date math or sorting on it correctly. `pd.to_datetime(..., errors="coerce")` parses each value into a proper `datetime64` date, and turns anything it *cannot* parse into `NaT` (missing) instead of crashing.

To measure real parsing failures (not just rows that were already blank), we compare how many values are missing **before** conversion vs. **after**:
- `original_missing` = rows where `last_review` was already empty in the source CSV
- `new_missing` = rows that are `NaT` after conversion
- `failed_conversions = new_missing - original_missing` = rows that *had* a value but pandas still could not turn it into a date (the true failure count).

In [ ]:
# Convert last_review from str to date, report how many rows fail
original_missing = df["last_review"].isna().sum()
df["last_review"] = pd.to_datetime(df["last_review"], errors="coerce")
new_missing = df["last_review"].isna().sum()
failed_conversions = new_missing - original_missing
print(f"Rows that failed date conversion (had a value but could not be parsed): {failed_conversions}")
print(f"Total missing last_review after conversion: {new_missing}")
df.dtypes

## 5. Clean the data
A quick audit (`df.isna().sum()`, `df.price.describe()`) shows the issues actually present in this dataset, and the cleaning below targets exactly those — nothing speculative:

| Issue found | Rows | Fix | Why |
|---|---|---|---|
| Exact duplicate rows | 0 | `drop_duplicates()` (safety net) | Confirmed none exist in step 3, but this keeps the notebook correct if the source file ever changes. |
| `price == 0` | 11 | drop | A nightly price of $0 is not a valid listing — it's a data-entry error, not a real value worth imputing. |
| `reviews_per_month` missing | 10,052 | fill with `0` | These are exactly the listings with no `last_review` — i.e. **zero reviews ever**, so 0 reviews/month is the true value, not missing data. |
| `last_review` missing (`NaT`) | 10,052 | leave as `NaT` | It correctly means "never reviewed". Filling it with a fake date would misrepresent listings as recently active. |
| `name` / `host_name` missing | 16 / 21 | fill with `"Unknown"` | Free-text labels, not used in any numeric analysis — filling avoids `NaN`-related errors (e.g. in groupby/display) without discarding otherwise-valid rows. |

Rows are only **dropped** when a value is objectively invalid (`price == 0`); everywhere else we **impute** because the missingness itself is meaningful (no reviews yet) or harmless (a text label).

In [ ]:
# --- Clean the data ---
before = len(df)

# 1. Drop exact duplicate rows (safety net; step 3 found 0)
df = df.drop_duplicates()

# 2. Drop rows with an invalid price of $0 — not a real listing value
df = df[df["price"] > 0]

# 3. No reviews yet -> reviews_per_month is genuinely 0, not missing
df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

# 4. Free-text fields: fill missing with a placeholder instead of dropping the row
df["name"] = df["name"].fillna("Unknown")
df["host_name"] = df["host_name"].fillna("Unknown")

# last_review stays NaT where there's no review — that IS the correct value

print(f"Rows before cleaning: {before}")
print(f"Rows after cleaning:  {len(df)}")
print(f"Rows dropped:         {before - len(df)}")
print()
print(df.isna().sum())

## 6. Save the cleaned data
Write the cleaned `df` to a new file, `AB_NYC_2019-clean.csv`, in the same folder as the original. The original `AB_NYC_2019.csv` is never modified — this only creates a new file.

In [ ]:
df.to_csv("AB_NYC_2019-clean.csv", index=False)
print("Saved cleaned data to AB_NYC_2019-clean.csv")